# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [33]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv()

import os, json
from google import genai

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (see .env.example)"
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"


## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [36]:
# from google.genai import types

# response = client.models.generate_content(
#     model=MODEL,
#     contents=[
#         "System: You are a concise support assistant.",
#         "User: Explain what a support ticket is in one sentence."
#     ]
# )

# print("Response:")
# print(response.text)

# print("\nToken Usage:")
# print(response.usage_metadata)

#### Note: Gemini API returned a 429 RESOURCE_EXHAUSTED error due to quota limitations. Following the lab instructions, Ollama (llama3.2:3b) was used as a local fallback.

In [38]:
from ollama import chat

response = chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "system",
            "content": "You are a concise support assistant."
        },
        {
            "role": "user",
            "content": "Explain what a support ticket is in one sentence."
        }
    ]
)

print(response["message"]["content"])

print(response)

A support ticket is an electronic request or inquiry submitted by a customer to a company's technical support team, usually via email, phone, or online portal, requesting assistance or resolution for an issue or problem with their product or service.
model='llama3.2:3b' created_at='2026-06-10T17:46:17.57593Z' done=True done_reason='stop' total_duration=1362643125 load_duration=163020375 prompt_eval_count=43 prompt_eval_duration=41917000 eval_count=47 eval_duration=1155627000 message=Message(role='assistant', content="A support ticket is an electronic request or inquiry submitted by a customer to a company's technical support team, usually via email, phone, or online portal, requesting assistance or resolution for an issue or problem with their product or service.", thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None


In [39]:
print("Prompt tokens:", response.get("prompt_eval_count"))
print("Response tokens:", response.get("eval_count"))

Prompt tokens: 43
Response tokens: 47


In [42]:
from ollama import chat

prompt = "Give a short slogan for a coffee shop."

for i in range(3):
    response = chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.1
        }
    )

    print(f"\nRun {i+1}")
    print(response["message"]["content"])


Run 1
Here are a few options:

1. "Brewing up happiness, one cup at a time."
2. "Fuel your day, one sip at a time."
3. "The perfect blend of flavor and community."
4. "Where every cup tells a story."
5. "Lift your spirits, one cup of coffee at a time."

Let me know if you'd like me to come up with more options!

Run 2
Here are a few options:

1. "Brewing up happiness, one cup at a time."
2. "Fuel your day, one sip at a time."
3. "The perfect blend of flavor and community."
4. "Where every cup tells a story."
5. "Lift your spirits, one cup of coffee at a time."

Let me know if you'd like me to come up with more options!

Run 3
Here are a few options:

1. "Brewing up moments"
2. "Fuel for the day ahead"
3. "The perfect cup, every time"
4. "Where every sip tells a story"
5. "Lift your morning, one cup at a time"

Let me know if you'd like me to come up with more options!


In [43]:
for i in range(3):
    response = chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 1.0
        }
    )

    print(f"\nRun {i+1}")
    print(response["message"]["content"])


Run 1
"Fuel Your Moment"

This slogan aims to capture the idea that coffee is not just a drink, but a moment of energy and inspiration to take on the day.

Run 2
Here are a few options:

1. "Brewing up happiness, one cup at a time."
2. "Fuel your day, one cup of love."
3. "The perfect blend, every time."
4. "Where every sip feels like home."
5. "Elevate your morning, elevate your life."

Let me know if you'd like me to come up with more options or if you have any specific preferences (e.g. short and catchy, punny, etc.)!

Run 3
Here are a few options:

1. "Brewing Up Happiness, One Cup at a Time."
2. "The Perfect Cup, Every Time."
3. "Where Coffee Meets Community."

Let me know if you'd like me to come up with more options!


**What changed, and why?**


I compared the outputs generated with temperature 0.1 and temperature 1.0 using the same prompt.

At temperature 0.1, the responses were very similar across runs. The model consistently produced the same structure and nearly identical slogans. This happened because a low temperature makes the model choose the highest-probability tokens, resulting in more deterministic and predictable outputs.

At temperature 1.0, the responses became more diverse. The model changed not only the wording of the slogans but also the response format. For example, one run produced a single slogan with an explanation, while other runs generated lists of different lengths. This happened because a higher temperature increases randomness during token sampling, allowing the model to explore a wider range of possible outputs.

Overall, lower temperature is useful when consistency and reliability are important, while higher temperature is better when creativity and variation are desired.


## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [44]:
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

print(tickets[:3])

[{'id': 1, 'text': 'I was charged twice for my subscription this month. Please refund the extra charge.'}, {'id': 2, 'text': 'The export button throws a 500 error every time I click it on the reports page.'}, {'id': 3, 'text': 'It would be great if you could add a dark mode to the dashboard.'}]


In [45]:
from ollama import chat

LABELS = ["billing", "bug", "feature_request", "other"]

def classify(text, style):

    if style == "zero_shot":

        prompt = f"""
Classify the following support ticket into exactly one label:

Labels:
billing
bug
feature_request
other

Ticket:
{text}

Return only the label.
"""

    elif style == "few_shot":

        prompt = f"""
Classify support tickets.

Ticket: I was charged twice this month.
Label: billing

Ticket: The app crashes when I click login.
Label: bug

Ticket: Please add dark mode support.
Label: feature_request

Ticket: {text}
Label:
"""

    elif style == "cot":

        prompt = f"""
Classify the support ticket.

Available labels:
billing, bug, feature_request, other

First briefly determine what the ticket is about.
Then choose the best label.

Ticket:
{text}

Format:

Reasoning: <short reasoning>
Label: <label>
"""

    response = chat(
        model="llama3.2:3b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={"temperature": 0.1}
    )

    output = response["message"]["content"].strip()

    # Label çıxarmaq
    for label in LABELS:
        if label in output.lower():
            return label

    return "unknown"

In [46]:
results = []

for ticket in tickets:

    text = ticket["text"]

    results.append({
        "ticket": text,
        "zero_shot": classify(text, "zero_shot"),
        "few_shot": classify(text, "few_shot"),
        "cot": classify(text, "cot")
    })

In [47]:
import pandas as pd

df = pd.DataFrame(results)

df

,ticket,zero_shot,few_shot,cot
0,I was charged twice for my subscription this m...,bug,billing,billing
1,The export button throws a 500 error every tim...,bug,billing,bug
2,It would be great if you could add a dark mode...,feature_request,billing,billing
3,How do I reset my password? I can't find the l...,bug,billing,billing
4,The app crashes on startup after the latest up...,bug,billing,bug
5,Can you send me a copy of my last invoice for ...,feature_request,billing,billing
6,Please add PDF export — CSV alone isn't enough...,feature_request,billing,feature_request
7,Just wanted to say the new UI looks really cle...,other,billing,billing


## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [51]:
import json
from ollama import chat

LABELS = ["billing", "bug", "feature_request", "other"]

def classify_structured(text):

    prompt = f"""
Classify the support ticket.

Allowed labels:
billing
bug
feature_request
other

Return ONLY valid JSON.

Requirements:
- label must be one of the allowed labels
- confidence must be a number between 0 and 1
- use higher confidence when the classification is obvious
- use lower confidence when the ticket is ambiguous

Example:

{{
  "label": "bug",
  "confidence": 0.95,
  "reason": "The application crashes during login."
}}

Ticket:
{text}
"""

    response = chat(
        model="llama3.2:3b",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        options={
            "temperature":0.1
        }
    )

    raw = response["message"]["content"]

    try:

        data = json.loads(raw)

        if data["label"] not in LABELS:
            raise ValueError("Invalid label")

        if not (0 <= data["confidence"] <= 1):
            raise ValueError("Invalid confidence")

        return data

    except Exception as e:

        return {
            "label":"other",
            "confidence":0.0,
            "reason":f"Invalid output: {e}"
        }

# TODO: run the SAME structured prompt against local Ollama (OpenAI-compatible
# endpoint at http://localhost:11434/v1) and note how reliably it returns valid JSON.


In [52]:
classify_structured(
    "The application crashes every time I try to login."
)

{'label': 'bug',
 'confidence': 0.9,
 'reason': 'The application crashes during login.'}

In [53]:
for ticket in tickets:

    result = classify_structured(
        ticket["text"]
    )

    print(result)

{'label': 'billing', 'confidence': 0.8, 'reason': 'Refund extra charge'}
{'label': 'bug', 'confidence': 0.9, 'reason': 'The export button throws a 500 error every time it is clicked on the reports page.'}
{'label': 'feature_request', 'confidence': 0.8, 'reason': 'add dark mode to the dashboard'}
{'label': 'other', 'confidence': 0.5, 'reason': 'User is unable to locate password reset link'}
{'label': 'bug', 'confidence': 0.9, 'reason': 'The app crashes on startup after the latest update on Android 14.'}
{'label': 'other', 'confidence': 0.5, 'reason': 'The request is not related to a specific issue or problem, but rather a general inquiry.'}
{'label': 'feature_request', 'confidence': 0.9, 'reason': 'Please add PDF export'}
{'label': 'other', 'confidence': 0.05, 'reason': 'The ticket does not contain any specific information about a bug or feature request.'}


**Local vs hosted: did the small local model produce valid JSON as reliably?**

The local Ollama model (llama3.2:3b) was able to consistently produce valid JSON outputs that matched the required schema. After improving the prompt, it also began generating meaningful `confidence` values instead of defaulting to 0.0, which shows better adherence to the structured format requirements.

However, while the JSON format was generally correct, the reliability is still not perfect. Some outputs showed less consistent reasoning quality and occasional variation in confidence calibration. This means that although the structure is stable, the semantic quality of the fields is not always fully reliable.

Compared to a hosted model with native structured-output support, the local model requires careful prompt engineering and validation logic to ensure correctness. Hosted models are typically more robust in following strict schemas and producing well-calibrated confidence scores without additional prompting effort.

Overall, the local model is now usable for structured output tasks, but it is still less reliable than hosted models in terms of consistency and semantic accuracy.
